# Fenotipado fisiológico multimodal de la obesidad
## Demostración reproducible con datos completamente simulados

**Área:** Ciencia de Datos e Inteligencia Artificial

> **Aviso:** todos los datos de este notebook son sintéticos. No corresponden a pacientes reales ni provienen de VIDANTE IPS.

### Flujo
1. Simulación de datos multimodales.
2. Control de calidad.
3. Estandarización.
4. PCA.
5. Clustering no supervisado.
6. Caracterización de fenotipos.
7. Clasificación supervisada.
8. Interpretabilidad.
9. Exportación reproducible.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
pd.set_option("display.max_columns", 50)
print("Entorno listo. Python", sys.version.split()[0])

## 1. Simulación de datos

Para demostrar el pipeline se generan tres **estructuras fisiológicas latentes simuladas**. Se usan solo para validar que el flujo analítico funciona. No representan fenotipos clínicos reales.


In [ ]:
n = 150
perfil_latente = rng.choice([0,1,2], size=n, p=[0.32,0.38,0.30])
sexo = rng.choice(["Mujer","Hombre"], size=n, p=[0.65,0.35])

def sim(means, sd, low=None, high=None):
    x = np.array([rng.normal(means[p], sd) for p in perfil_latente])
    if low is not None: x = np.maximum(x, low)
    if high is not None: x = np.minimum(x, high)
    return x

edad = sim([40,45,49], 8, 18, 70)
imc = sim([32.2,35.2,39.0], 2.0, 30, 50)
grasa_pct = sim([34,41,48], 4, 25, 60)
masa_grasa_kg = sim([31,39,49], 6, 18, 75)
masa_muscular_kg = sim([31,29,27], 4.5, 18, 48)
grasa_visceral = sim([10,15,20], 2.5, 5, 30)
cintura_cm = sim([99,110,124], 7, 82, 155)

rmssd = sim([46,31,19], 6, 7, 80)
sdnn = sim([58,43,29], 7, 10, 100)
hf_power = sim([700,430,230], 110, 40, 1300)
sd1 = rmssd/np.sqrt(2) + rng.normal(0,1.5,n)
sd2 = 1.25*sdnn + rng.normal(0,4,n)

smo2_baseline = sim([68,64,60], 4, 40, 85)
smo2_min = sim([39,36,34], 4, 15, 70)
delta_smo2 = smo2_baseline - smo2_min
pendiente_desaturacion = sim([-0.85,-1.05,-1.25], 0.12, -2.0, -0.2)
pendiente_reoxigenacion = sim([1.80,1.30,0.85], 0.18, 0.2, 3.0)
tiempo_recuperacion = sim([42,61,86], 9, 20, 150)
espesor_adiposo_mm = sim([13,20,29], 3, 5, 45)

df = pd.DataFrame({
    "edad": edad.round(1), "sexo": sexo, "IMC": imc.round(2),
    "grasa_pct": grasa_pct.round(2), "masa_grasa_kg": masa_grasa_kg.round(2),
    "masa_muscular_kg": masa_muscular_kg.round(2), "grasa_visceral": grasa_visceral.round(2),
    "cintura_cm": cintura_cm.round(1), "RMSSD_ms": rmssd.round(2),
    "SDNN_ms": sdnn.round(2), "HF_power": hf_power.round(2),
    "SD1_ms": sd1.round(2), "SD2_ms": sd2.round(2),
    "SmO2_baseline": smo2_baseline.round(2), "SmO2_min": smo2_min.round(2),
    "Delta_SmO2": delta_smo2.round(2),
    "pendiente_desaturacion": pendiente_desaturacion.round(3),
    "pendiente_reoxigenacion": pendiente_reoxigenacion.round(3),
    "tiempo_recuperacion_s": tiempo_recuperacion.round(1),
    "espesor_adiposo_mm": espesor_adiposo_mm.round(1),
    "perfil_latente_simulado": perfil_latente
})
print("Dimensión:", df.shape)
display(df.head())

## 2. Control de calidad y exploración

In [ ]:
display(df.isna().sum().to_frame("n_missing"))
display(df.describe(numeric_only=True).T.round(2))

## 3. Selección de características multimodales

In [ ]:
features = [
    "grasa_pct","masa_grasa_kg","masa_muscular_kg","grasa_visceral","cintura_cm",
    "RMSSD_ms","SDNN_ms","HF_power","SD1_ms","SD2_ms",
    "SmO2_baseline","SmO2_min","Delta_SmO2","pendiente_desaturacion",
    "pendiente_reoxigenacion","tiempo_recuperacion_s","espesor_adiposo_mm"
]
X = df[features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Matriz analítica:", X_scaled.shape)

## 4. Reducción de dimensionalidad con PCA

In [ ]:
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)
varianza_acumulada = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(7,4))
plt.plot(range(1,len(varianza_acumulada)+1), varianza_acumulada, marker="o")
plt.axhline(0.80, linestyle="--")
plt.xlabel("Número de componentes")
plt.ylabel("Varianza explicada acumulada")
plt.title("PCA: varianza explicada acumulada")
plt.tight_layout()
plt.show()

pca_2 = PCA(n_components=2)
X_pca_2 = pca_2.fit_transform(X_scaled)
df["PC1"], df["PC2"] = X_pca_2[:,0], X_pca_2[:,1]
print("Varianza PC1+PC2:", round(pca_2.explained_variance_ratio_.sum(),3))

## 5. Selección exploratoria del número de clusters

Se comparan soluciones entre 2 y 6 grupos mediante **silhouette score** y **Davies–Bouldin**.


In [ ]:
res = []
for k in range(2,7):
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=50)
    labels = model.fit_predict(X_scaled)
    res.append({
        "k": k,
        "silhouette": silhouette_score(X_scaled, labels),
        "davies_bouldin": davies_bouldin_score(X_scaled, labels)
    })
metricas_cluster = pd.DataFrame(res)
display(metricas_cluster.round(3))

plt.figure(figsize=(7,4))
plt.plot(metricas_cluster["k"], metricas_cluster["silhouette"], marker="o")
plt.xlabel("Número de clusters (k)")
plt.ylabel("Silhouette score")
plt.title("Selección exploratoria de k")
plt.tight_layout()
plt.show()

## 6. Fenotipado no supervisado

Para la demostración se fija `k=3`. En el estudio real se combinarán criterios estadísticos, estabilidad e interpretabilidad fisiológica.


In [ ]:
kmeans = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=50)
df["fenotipo_modelo"] = kmeans.fit_predict(X_scaled)

display(df["fenotipo_modelo"].value_counts().sort_index().to_frame("n"))
ari = adjusted_rand_score(df["perfil_latente_simulado"], df["fenotipo_modelo"])
print("Adjusted Rand Index frente al patrón latente simulado:", round(ari,3))

plt.figure(figsize=(7,5))
for cluster in sorted(df["fenotipo_modelo"].unique()):
    temp = df[df["fenotipo_modelo"] == cluster]
    plt.scatter(temp["PC1"], temp["PC2"], label=f"Fenotipo {cluster}", alpha=0.7)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Fenotipos simulados en espacio PCA")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Caracterización fisiológica de los fenotipos

In [ ]:
perfil_fenotipos = df.groupby("fenotipo_modelo")[features + ["IMC","edad"]].mean().round(2)
display(perfil_fenotipos)

perfil_z = (
    pd.DataFrame(X_scaled, columns=features)
    .assign(fenotipo_modelo=df["fenotipo_modelo"].values)
    .groupby("fenotipo_modelo").mean().T
)
display(perfil_z.round(2))

### Nota interpretativa

Un cluster no equivale automáticamente a un fenotipo clínico. En datos reales debe demostrar estabilidad, coherencia fisiológica, tamaño suficiente y reproducibilidad.


## 8. Clasificador supervisado

In [ ]:
X_model = df[features]
y = df["fenotipo_modelo"]

X_train, X_test, y_train, y_test = train_test_split(
    X_model, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

rf = RandomForestClassifier(
    n_estimators=500, random_state=RANDOM_STATE, class_weight="balanced"
)
rf.fit(X_train, y_train)
pred = rf.predict(X_test)

print(classification_report(y_test, pred))
ConfusionMatrixDisplay.from_predictions(y_test, pred)
plt.title("Matriz de confusión")
plt.tight_layout()
plt.show()

## 9. Interpretabilidad mediante importancia de variables

In [ ]:
importancias = (
    pd.DataFrame({"variable": features, "importancia": rf.feature_importances_})
    .sort_values("importancia", ascending=False)
    .reset_index(drop=True)
)
display(importancias)

top = importancias.head(12).sort_values("importancia")
plt.figure(figsize=(8,6))
plt.barh(top["variable"], top["importancia"])
plt.xlabel("Importancia relativa")
plt.title("Variables con mayor contribución")
plt.tight_layout()
plt.show()

## 10. Validación cruzada

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(rf, X_model, y, cv=cv, scoring="balanced_accuracy")
print("Balanced accuracy por fold:", np.round(scores,3))
print("Promedio:", round(scores.mean(),3))
print("DE:", round(scores.std(),3))

## 11. Comparación descriptiva con IMC

Se explora si los fenotipos multimodales muestran diferencias que no pueden resumirse solo con el IMC.


In [ ]:
display(
    df.groupby("fenotipo_modelo")["IMC"]
      .agg(["mean","std","min","max","count"])
      .round(2)
)

## 12. Exportación reproducible

In [ ]:
cwd = Path.cwd()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
data_out = repo_root / "data" / "simulated"
results_out = repo_root / "results"
data_out.mkdir(parents=True, exist_ok=True)
results_out.mkdir(parents=True, exist_ok=True)

df.to_csv(data_out / "datos_simulados_fenotipado_obesidad.csv", index=False)
perfil_fenotipos.to_csv(results_out / "perfil_fenotipos_simulados.csv")
metricas_cluster.to_csv(results_out / "metricas_cluster.csv", index=False)
importancias.to_csv(results_out / "importancia_variables.csv", index=False)

print("Exportación completada.")

# Conclusión

Este notebook demuestra un pipeline:

**datos multimodales → control de calidad → estandarización → PCA → clustering → fenotipos → clasificación → interpretabilidad**

### Alcance
- Datos completamente simulados.
- No contiene resultados clínicos reales.
- Los patrones latentes fueron introducidos deliberadamente para verificar el funcionamiento del pipeline.
- El protocolo estadístico definitivo deberá fijarse antes del análisis de la cohorte real.
